# Notebook 4: Simulation-Based Inference (SBI)

In this notebook, we will introduce **Simulation-Based Inference (SBI)**, also known as Likelihood-Free Inference (LFI).

## Concept

In many areas of science (like astrophysics), we have a complex simulation (forward model) that predicts data $x$ given parameters $\theta$. However, the likelihood $p(x|\theta)$ is often intractable or too expensive to compute.

SBI allows us to infer the posterior $p(\theta|x)$ using neural networks trained on pairs of $(\theta, x)$ generated by the simulator.

## Methods

We will explore two common techniques provided by the `sbi` package:
1.  **NPE (Neural Posterior Estimation)**: Directly learns $p(\theta|x)$ using a Normalizing Flow.
2.  **NLE (Neural Likelihood Estimation)**: Learns $p(x|\theta)$ and then uses MCMC to sample from the posterior.

## Task: Gaussian Toy Model

We will infer the mean and variance of a Gaussian distribution.

In [ ]:
import torch
import matplotlib.pyplot as plt
from sbi.inference import SNPE, SNLE, simulate_for_sbi
from sbi.utils import process_prior, process_simulator
from sbi import utils as utils
from sbi import analysis as analysis

# 1. Define Simulator
def simulator(theta):
    # theta: [mean, log_std]
    # Returns a sample x of size 10
    mu = theta[0]
    sigma = torch.exp(theta[1])
    return torch.normal(mu, sigma, size=(10,))

# 2. Define Prior
# Uniform prior over mean [-5, 5] and log_std [-2, 1]
prior = utils.BoxUniform(low=torch.tensor([-5.0, -2.0]), 
                         high=torch.tensor([5.0, 1.0]))

# Check simulator
theta_test = torch.tensor([0.0, 0.0]) # mean=0, std=1
x_test = simulator(theta_test)
print("Sample theta:", theta_test)
print("Sample x:", x_test)

## Neural Posterior Estimation (NPE)

NPE trains a neural density estimator (usually a MAF - Masked Autoregressive Flow) to approximate the posterior directly.

In [ ]:
# Prepare simulator and prior
prior = process_prior(prior)[0]
simulator = process_simulator(simulator, prior, is_numpy_simulator=False)

# Instantiate NPE Inference object
# density_estimator='maf' is the default
inference = SNPE(prior=prior, density_estimator='maf')

# Generate Training Data
# We simulate 2000 pairs of (theta, x)
theta, x = simulate_for_sbi(simulator, proposal=prior, num_simulations=2000)

# Append simulations to inference object
inference = inference.append_simulations(theta, x)

# Train density estimator
density_estimator = inference.train()

# Build posterior
posterior = inference.build_posterior(density_estimator)

### Inference on Observation

Let's assume we observe a dataset $x_{obs}$ generated from a known $\theta_{true}$.

In [ ]:
# True parameters
theta_true = torch.tensor([2.0, -0.5]) # mean=2, std=exp(-0.5) approx 0.6
# prepare_for_sbi wraps the simulator, so it might expect batched theta
x_obs = simulator(theta_true.unsqueeze(0))

# Sample from learned posterior
samples = posterior.sample((5000,), x=x_obs)

# Visualize
fig, axes = analysis.pairplot(
    samples,
    points=theta_true,
    labels=['Mean', 'Log Std'],
    limits=[[-5, 5], [-2, 1]],
    fig_kwargs={'points_colors': 'r',
               'title': 'NPE Posterior'}
)

## Neural Likelihood Estimation (NLE)

NLE trains a neural density estimator to approximate the likelihood $p(x|\theta)$. We then use MCMC to obtain posterior samples.

In [ ]:
# Instantiate NLE Inference object
inference_nle = SNLE(prior=prior, density_estimator='maf')

# Append same simulations
inference_nle = inference_nle.append_simulations(theta, x)

# Train
density_estimator_nle = inference_nle.train()

# Build posterior (this involves MCMC sampling under the hood when sampling)
posterior_nle = inference_nle.build_posterior(density_estimator_nle)

# Sample
# MCMC can be slower
samples_nle = posterior_nle.sample((5000,), x=x_obs)

# Visualize
fig, axes = analysis.pairplot(
    samples_nle, 
    points=theta_true,
    labels=['Mean', 'Log Std'],
    limits=[[-5, 5], [-2, 1]],
    fig_kwargs={'points_colors': 'r',
               'title': 'NLE Posterior'}
)

## Coverage Tests (TARP)

It is crucial to verify if the inferred posterior is well-calibrated. A common way to check this is **Expected Coverage Probability**. If a region contains $x\%$ of the posterior probability mass, it should contain the true parameter $x\%$ of the time.

We will use the **TARP (Test for Accuracy of Reservoir Predictions)** method. TARP checks coverage by computing the probability mass of the credibility region (reservoir) associated with the true parameter for many different simulations.

1. Generate a test set of $\theta_{test} \sim p(\theta)$ and corresponding $x_{test}$.
2. For each pair, sample from the estimated posterior $p(\theta|x_{test})$.
3. Compute the fraction of posterior samples closer to the posterior mean (or peak) than the true $\theta_{test}$.
4. Plot the cumulative distribution of these fractions (P-P plot).

In [ ]:
from tarp import get_tarp_coverage

# 1. Generate Test Set
num_test = 500
theta_test, x_test = simulate_for_sbi(simulator, proposal=prior, num_simulations=num_test)

# 2. Sample from Posterior for each x_test
# We need a batch of posterior samples for each x in x_test.
# NPE allows batch sampling directly.

num_samples = 5000
posterior_samples = []

# Sampling loop (can be slow for many test points)
for i in range(num_test):
    # Sample from posterior given x_test[i]
    # x_test[i] needs batch dim [1, 10]
    # The output sample s has shape [num_samples, num_params]
    s = posterior.sample((num_samples,), x=x_test[i].unsqueeze(0), show_progress_bars=False)
    posterior_samples.append(s)

# Stack: [num_test, num_samples, num_params]
posterior_samples = torch.stack(posterior_samples)

# TARP expects samples to be [num_samples, num_sims, num_dims]
# Currently posterior_samples is [num_test (sims), num_samples, num_params]
# So we transpose to [num_samples, num_test, num_params]
posterior_samples = posterior_samples.transpose(0, 1)

# 3. Compute Coverage using TARP
# theta_test: [num_test, num_params]
# posterior_samples: [num_samples, num_test, num_params]
ecp, alpha = get_tarp_coverage(posterior_samples.numpy(), theta_test.numpy(), references='random', metric='euclidean')

# 4. Plot P-P Plot
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Ideal')
plt.plot(alpha, ecp, label='NPE')
plt.xlabel('Credibility Level')
plt.ylabel('Expected Coverage')
plt.title('TARP Coverage Test')
plt.legend()
plt.show()